<a href="https://colab.research.google.com/github/sebenemaryamashebir-cmd/cassava-disease-detector/blob/Seben/Evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

"""
Model Evaluation

Loads the checkpoint saved on train.py (checkpoints/best_model.pt),
re-creates the EXACT SAME test split previously used (same seed, same
random_split call), runs the model on it, and reports:

  - Overall accuracy
  - Per-class precision, recall, F1-score
  - Macro-average and weighted-average precision/recall/F1
  - Confusion matrix (printed + saved as a PNG heatmap)
  - A saved text report (classification_report.txt)

IMPORTANT: this script re-derives the test split the same way train.py does
(same data_dir, same manual_seed(42), same 80/10/10 sizes) so that "test_loader"
here refers to images the model never trained or validated on. If you use a
different data_dir or seed than train.py did, this split will NOT match the
one used during training and your test numbers will be meaningless.

Run:
    python evaluate.py --data-dir /path/to/cassava_dataset \
                        --checkpoint checkpoints/best_model.pt
"""
import argparse # allow commands without changing the code
import os # for folder operations

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms

# sklearn gives us precision/recall/F1/confusion matrix in one place instead
# of hand-rolling them from scratch.
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns

from model import build_model # import the CNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IMG_SIZE = 224
BATCH_SIZE = 32

ModuleNotFoundError: No module named 'model'

In [ ]:
# Same "no augmentation" transform used for val/test: augmentation
# (flip/rotate/jitter) is only for training. Evaluating on augmented images
# would give a distorted, unrealistic picture of real-world performance.
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
# TransformSubset
class TransformSubset(Dataset):
    """Same wrapper class train.py uses: applies a transform to a Subset
    of the base ImageFolder so we can give the test split its own transform."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx): # getting one image
        image, label = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


In [ ]:
# build_test_loader
def build_test_loader(data_dir, batch_size=BATCH_SIZE):
    """Recreate the identical 80/10/10 split train.py made, then pull out
    only the test portion. Using the same seed (42) and same split sizes
    guarantees these are the same held-out images Member 2 never trained on."""
    base_dataset = datasets.ImageFolder(root=data_dir)

    # These class names come straight from the folder names on disk, so this
    # is the ground truth for how many classes actually exist -- use THIS,
    # not model.py's hardcoded CLASS_NAMES, to avoid the CBB/CBSD/CMD/Healthy
    # (4-class) vs. actual 5-class (+green_mottle) mismatch mentioned above.
    class_names = [name.replace("Cassava___", "") for name in base_dataset.classes]
    num_classes = len(class_names)
    print("Classes found on disk:", class_names)

    train_size = int(0.80 * len(base_dataset))
    val_size = int(0.10 * len(base_dataset))
    test_size = len(base_dataset) - train_size - val_size

    generator = torch.Generator().manual_seed(42)  # must match train.py exactly
    _, _, test_split = random_split(
        base_dataset, [train_size, val_size, test_size], generator=generator
    )

    test_dataset = TransformSubset(test_split, eval_transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    print(f"Test set size: {len(test_split)}")
    return test_loader, class_names, num_classes


In [ ]:
# run_inference
@torch.no_grad()  # no gradients needed for pure evaluation -- saves memory/time
def run_inference(model, loader):
    """Feed every test batch through the model and collect predictions
    alongside the true labels, so we can compute metrics on the full set
    at once rather than batch-by-batch."""
    model.eval()  # turns off dropout so predictions are deterministic

    all_preds = [] # stores predictions
    all_labels = [] # stores correct answers
    all_probs = []  # kept in case you want ROC/AUC or top-k analysis later

    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)                     # raw logits, shape [batch, num_classes]
        probs = torch.softmax(outputs, dim=1)        # convert logits -> probabilities
        preds = outputs.argmax(dim=1)                # predicted class = highest logit

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
        all_probs.append(probs.cpu().numpy())

    return (
        np.concatenate(all_preds),
        np.concatenate(all_labels),
        np.concatenate(all_probs),
    )

In [ ]:
# compute_metrics and print_metrics
def compute_metrics(y_true, y_pred, class_names):
    """Compute accuracy plus per-class and averaged precision/recall/F1."""
    accuracy = accuracy_score(y_true, y_pred)

    # per-class precision/recall/F1 (one number per class)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )

    # macro avg = simple mean across classes (treats every class equally,
    # regardless of size -- good for spotting how badly the minority class,
    # bacterial_blight, is doing)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    # weighted avg = mean weighted by class support (closer to overall
    # accuracy since it's dominated by the majority class, mosaic_disease)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "per_class": {
            class_names[i]: {
                "precision": precision[i],
                "recall": recall[i],
                "f1": f1[i],
                "support": int(support[i]),
            }
            for i in range(len(class_names))
        },
        "macro": {"precision": precision_macro, "recall": recall_macro, "f1": f1_macro},
        "weighted": {"precision": precision_weighted, "recall": recall_weighted, "f1": f1_weighted},
    }


def print_metrics(metrics):
    print(f"\nOverall accuracy: {metrics['accuracy']:.4f}\n")

    print(f"{'Class':<20}{'Precision':>10}{'Recall':>10}{'F1':>10}{'Support':>10}")
    for cls, vals in metrics["per_class"].items():
        print(f"{cls:<20}{vals['precision']:>10.3f}{vals['recall']:>10.3f}"
              f"{vals['f1']:>10.3f}{vals['support']:>10d}")

    print(f"\n{'Macro avg':<20}{metrics['macro']['precision']:>10.3f}"
          f"{metrics['macro']['recall']:>10.3f}{metrics['macro']['f1']:>10.3f}")
    print(f"{'Weighted avg':<20}{metrics['weighted']['precision']:>10.3f}"
          f"{metrics['weighted']['recall']:>10.3f}{metrics['weighted']['f1']:>10.3f}")



In [ ]:
# plot_confusion_matrix
def plot_confusion_matrix(y_true, y_pred, class_names, out_path):
    """Save a heatmap of the confusion matrix. This is what lets us check
    the hypothesis from the EDA -- that bacterial_blight gets
    confused with brown_streak, and green_mottle with mosaic_disease. (from pre-training)"""
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
    )
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title("Confusion Matrix — Test Set")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"\nConfusion matrix saved to {out_path}")
    return cm


In [ ]:
# Main Program
# Colab equivalent of the argparse args in the original script
DATA_DIR = "/content/cassava_dataset"
CHECKPOINT = "checkpoints/best_model.pt"
OUT_DIR = "eval_results"

os.makedirs(OUT_DIR, exist_ok=True)

# 1. Rebuild the exact held-out test set
test_loader, class_names, num_classes = build_test_loader(DATA_DIR)

# 2. Load the trained model
model = build_model(num_classes=num_classes).to(device)
state_dict = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(state_dict)
print(f"Loaded checkpoint from {CHECKPOINT}")

# 3. Run inference on the test set
y_pred, y_true, _ = run_inference(model, test_loader)

# 4. Compute + print accuracy/precision/recall/F1
metrics = compute_metrics(y_true, y_pred, class_names)
print_metrics(metrics)

# 5. Confusion matrix (printed + saved as an image)
cm = plot_confusion_matrix(y_true, y_pred, class_names,
                            os.path.join(OUT_DIR, "confusion_matrix.png"))
print("\nRaw confusion matrix:\n", cm)

# 6. Save a full sklearn text report too, for the writeup/documentation
report_text = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
report_path = os.path.join(OUT_DIR, "classification_report.txt")
with open(report_path, "w") as f:
    f.write(f"Checkpoint: {CHECKPOINT}\n")
    f.write(f"Overall accuracy: {metrics['accuracy']:.4f}\n\n")
    f.write(report_text)
print(f"Classification report saved to {report_path}")